# LLM 06: Decoding Strategies

Hands-on:
1. Implement greedy, top-k, top-p, temperature sampling from scratch
2. Visualize how temperature reshapes a distribution
3. Use HuggingFace `generate()` with different configs
4. Demonstrate structured decoding with a simple regex mask
5. Exercises: speculative decoding stub, determinism check

Deps: `pip install torch matplotlib numpy`

## 1. The Core Sampling Primitives

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

def greedy(logits):
    return int(logits.argmax())

def sample_temperature(logits, T=1.0):
    probs = F.softmax(logits / T, dim=-1)
    return int(torch.multinomial(probs, 1))

def sample_top_k(logits, k=40, T=1.0):
    logits = logits.clone()
    v, _ = torch.topk(logits, k)
    logits[logits < v[-1]] = float('-inf')
    return sample_temperature(logits, T)

def sample_top_p(logits, p=0.9, T=1.0):
    logits = logits.clone() / T
    sorted_logits, idx = torch.sort(logits, descending=True)
    cum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    mask = cum > p
    mask[1:] = mask[:-1].clone()   # keep first token above threshold
    mask[0] = False
    sorted_logits[mask] = float('-inf')
    probs = F.softmax(sorted_logits, dim=-1)
    return int(idx[torch.multinomial(probs, 1)])

# fake logits for a 10-token vocab
logits = torch.tensor([3.0, 2.8, 2.5, 1.9, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0])
print('greedy    :', greedy(logits))
print('T=1.0     :', [sample_temperature(logits, 1.0) for _ in range(10)])
print('T=0.3     :', [sample_temperature(logits, 0.3) for _ in range(10)])
print('top_k=3   :', [sample_top_k(logits, 3) for _ in range(10)])
print('top_p=0.6 :', [sample_top_p(logits, 0.6) for _ in range(10)])

## 2. Temperature Reshapes the Distribution

In [ ]:
import matplotlib.pyplot as plt

temps = [0.3, 0.7, 1.0, 1.5, 2.0]
fig, axes = plt.subplots(1, len(temps), figsize=(18, 3), sharey=True)
for ax, T in zip(axes, temps):
    p = F.softmax(logits / T, dim=-1)
    ax.bar(range(len(p)), p.numpy())
    ax.set_title(f'T={T}')
    ax.set_xlabel('token id')
axes[0].set_ylabel('P(token)')
plt.tight_layout()
plt.show()

## 3. Top-k vs Top-p Candidate Sets

In [ ]:
def candidates_top_k(logits, k):
    return torch.topk(logits, k).indices.tolist()

def candidates_top_p(logits, p):
    sorted_logits, idx = torch.sort(logits, descending=True)
    cum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    keep = (cum <= p).nonzero().flatten().tolist()
    if not keep:
        keep = [0]
    return idx[:max(keep)+1].tolist()

# a peaked distribution (model confident)
peaked = torch.tensor([5.0, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5, -2.0, -2.5, -3.0])
# a flat distribution (model uncertain)
flat = torch.tensor([1.0, 0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6, 0.55, 0.5])

for name, l in [('peaked', peaked), ('flat', flat)]:
    print(f'\n{name} distribution:')
    print(f'  top_k=5  -> candidates: {candidates_top_k(l, 5)}')
    print(f'  top_p=0.9 -> candidates: {candidates_top_p(l, 0.9)}')

Notice: top-p adapts. When the model is confident, it keeps 1-2 tokens; when uncertain, it keeps many. Top-k is rigid.

## 4. Structured Decoding: Regex-Masked Sampling

Force output to match a regex by zeroing out logits of tokens that would violate the pattern. This is a simplified version of what Outlines / LM Format Enforcer do in production.

In [ ]:
import re

# A 10-token vocab for toy example
vocab = ['{', '"', 'n', 'a', 'm', 'e', ':', ' ', 'x', '}']

def regex_mask(partial, pattern, vocab, logits):
    masked = logits.clone()
    for i, tok in enumerate(vocab):
        if re.fullmatch(pattern, partial + tok) is None and not re.match(pattern, partial + tok):
            masked[i] = float('-inf')
    return masked

# pattern: any prefix of a minimal JSON-like object
pattern = r'\{"name": "[a-z]+"\}?'

partial = ''
logits_all = torch.tensor([0.1]*10)   # uniform baseline
for _ in range(12):
    masked = regex_mask(partial, pattern, vocab, logits_all)
    if torch.isinf(masked).all():
        break
    nxt = int(F.softmax(masked, dim=-1).multinomial(1))
    partial += vocab[nxt]
print('generated:', partial)

## 5. End-to-End: HuggingFace `generate()` with Different Configs

Use a tiny model to see real decoding in action.

In [ ]:
try:
    from transformers import GPT2LMHeadModel, GPT2Tokenizer
    tok = GPT2Tokenizer.from_pretrained('gpt2')
    model = GPT2LMHeadModel.from_pretrained('gpt2')

    prompt = 'The secret to a productive day is'
    ids = tok.encode(prompt, return_tensors='pt')

    configs = [
        dict(do_sample=False),                                # greedy
        dict(num_beams=4, early_stopping=True),                # beam
        dict(do_sample=True, temperature=1.0, top_k=40),       # top-k
        dict(do_sample=True, temperature=1.0, top_p=0.9),      # top-p
        dict(do_sample=True, temperature=0.3, top_p=1.0),      # cold
        dict(do_sample=True, temperature=1.4, top_p=0.95),     # hot
    ]

    for cfg in configs:
        out = model.generate(ids, max_new_tokens=25, pad_token_id=tok.eos_token_id, **cfg)
        text = tok.decode(out[0], skip_special_tokens=True)
        print(f'{str(cfg):<80}  ->  {text[len(prompt):]!r}')
except Exception as e:
    print('Skipped HF demo (no transformers?):', e)

## 6. Exercise: Speculative Decoding Stub

Sketch the core algorithm:

```python
def speculative_decode(draft, target, prompt_ids, k=4, max_new=50):
    ids = list(prompt_ids)
    while len(ids) < max_new:
        draft_ids = [sample(draft(ids + draft_ids)) for _ in range(k)]
        # verify all k with one parallel forward of target
        target_logits = target(ids + draft_ids)
        # accept/reject per token; fall back at first rejection
        ...
    return ids
```

**Try it:** use two GPT-2 variants (`gpt2` as draft, `gpt2-large` as target). Measure tokens/second vs running `gpt2-large` alone. Target ~2x speedup.

## 7. Exercise: Determinism Check

Run greedy generation 10 times on the same prompt. Confirm bit-identical outputs.

Then run sampling with `temperature=0.7, seed=42` 10 times. Are they identical? Try varying batch size at inference — does that change the output?

## 8. Takeaways

- **Sampling = how the distribution becomes text.** Understand the knobs.
- **Top-p beats top-k** because it adapts to the model's confidence.
- **Pick settings per feature, not per model.** Factual Q&A ≠ creative chat.
- **Structured decoding is free accuracy** when output shape is known.
- **`temperature=0`** is approximately but not always bit-deterministic on GPUs.

Next: **07 — Prompt Engineering (Intro)** — how the text before the decode shapes what gets decoded.